# Azure ML Capstone Project - Rubric Validation Notebook

This notebook validates the Capstone project against the Udacity rubric criteria.

**Project**: Heart Failure Prediction with AutoML + HyperDrive
**Date**: 2026-06-18
**Status**: Validation Framework Ready for Execution

## 1. Environment Setup and Azure ML Workspace Connection

In [ ]:
# Import required libraries
import os
import json
import requests
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np

# Azure ML imports
from azureml.core import Workspace, Experiment, Dataset, Model
from azureml.core.authentication import InteractiveLoginAuthentication, AzureCliAuthentication, ServicePrincipalAuthentication
from azureml.train.automl import AutoMLConfig
from azureml.train.hyperdrive import HyperDriveConfig
from azureml.core.webservice import Webservice
from azureml.widgets import RunDetails

print("✓ All libraries imported successfully")

In [ ]:
# Connect to Azure ML Workspace
config_path = Path("../config/aml_config.capstone.json")

try:
    ws = Workspace.from_config(path=str(config_path))
    print(f"✓ Connected to workspace: {ws.name}")
    print(f"  Subscription: {ws.subscription_id}")
    print(f"  Resource Group: {ws.resource_group}")
except Exception as e:
    print(f"✗ Workspace connection failed: {e}")
    ws = None

# Project root and artifact paths
project_root = Path("..").resolve()
artifacts_dir = project_root / "artifacts"
screenshots_dir = project_root / "screenshots"

print(f"\n✓ Project root: {project_root}")
print(f"✓ Artifacts dir: {artifacts_dir}")
print(f"✓ Screenshots dir: {screenshots_dir}")

## 2. Rubric Criteria as Structured Validation Rules

In [ ]:
# Define rubric criteria as structured validation checklist
rubric_criteria = {
    "PROJECT_SETUP": {
        "External Dataset": {
            "required": True,
            "evidence": "Dataset registered in workspace from external source (not Azure built-in)",
            "status": None
        },
        "README Exists": {
            "required": True,
            "evidence": "README.md in project root",
            "status": None
        },
        "README: Project Overview": {
            "required": True,
            "evidence": "Project overview section in README",
            "status": None
        },
        "README: Dataset Description": {
            "required": True,
            "evidence": "Dataset source, size, features, target variable documented",
            "status": None
        },
        "README: Data Ingestion Method": {
            "required": True,
            "evidence": "How dataset was uploaded to Azure ML workspace",
            "status": None
        },
        "README: AutoML Settings Explanation": {
            "required": True,
            "evidence": "Written explanation of AutoML config (not copy-pasted code)",
            "status": None
        },
        "README: HyperDrive Parameters Explanation": {
            "required": True,
            "evidence": "Written explanation of hyperparameter search space and ranges",
            "status": None
        },
        "README: Model Comparison": {
            "required": True,
            "evidence": "Comparison of AutoML vs HyperDrive models with metrics",
            "status": None
        },
        "README: Endpoint Instructions": {
            "required": True,
            "evidence": "Deployed endpoint URI and sample request/response examples",
            "status": None
        },
        "README: Future Improvements": {
            "required": True,
            "evidence": "Section outlining how to improve the project",
            "status": None
        },
        "README: Screenshots": {
            "required": True,
            "evidence": "7+ screenshots with descriptions",
            "status": None
        },
        "README: Screencast Link": {
            "required": True,
            "evidence": "YouTube or streaming service link with video demonstration",
            "status": None
        },
        "Screencast Video": {
            "required": True,
            "evidence": "1-5 min, 1080p+, 16:9, clear audio, working model demo + endpoint test",
            "status": None
        }
    },
    "AUTOML_MODEL": {
        "AutoML Code": {
            "required": True,
            "evidence": "src/automl_run.py with AutoMLConfig",
            "status": None
        },
        "AutoML Configuration": {
            "required": True,
            "evidence": "Task type, primary metric, cross-validation, timeout settings",
            "status": None
        },
        "RunDetails Widget": {
            "required": True,
            "evidence": "Screenshot showing AutoML progress and run details",
            "status": None
        },
        "Best Model Properties": {
            "required": True,
            "evidence": "Notebook displays best model metrics and parameters",
            "status": None
        },
        "Model Registration": {
            "required": True,
            "evidence": "Best model registered with run ID and version",
            "status": None
        }
    },
    "HYPERDRIVE_MODEL": {
        "HyperDrive Code": {
            "required": True,
            "evidence": "src/hyperdrive_run.py with HyperDriveConfig",
            "status": None
        },
        "Sampling Method": {
            "required": True,
            "evidence": "Grid/Random/Bayesian sampling defined",
            "status": None
        },
        "Metric Logging": {
            "required": True,
            "evidence": "Training script logs metrics (accuracy, etc.)",
            "status": None
        },
        "Early Termination Policy": {
            "required": False,  # Not required for Bayesian
            "evidence": "Early stopping policy configured",
            "status": None
        },
        "Hyperparameter Count": {
            "required": True,
            "evidence": "At least 2 hyperparameters tuned",
            "status": None
        },
        "RunDetails Widget": {
            "required": True,
            "evidence": "Screenshot showing HyperDrive progress",
            "status": None
        },
        "Best Hyperparameters": {
            "required": True,
            "evidence": "Best run with tuned hyperparameter values displayed",
            "status": None
        },
        "Model Registration": {
            "required": True,
            "evidence": "Best HyperDrive model registered with run ID",
            "status": None
        }
    },
    "DEPLOYMENT": {
        "Model Registration": {
            "required": True,
            "evidence": "Model registered in workspace",
            "status": None
        },
        "Model Deployment": {
            "required": True,
            "evidence": "Model deployed to ACI or AKS",
            "status": None
        },
        "Environment File": {
            "required": True,
            "evidence": "Environment configuration with dependencies",
            "status": None
        },
        "Endpoint Active": {
            "required": True,
            "evidence": "Screenshot showing endpoint state = Active",
            "status": None
        },
        "Inference Request Code": {
            "required": True,
            "evidence": "HTTP request code sending JSON to endpoint",
            "status": None
        },
        "Inference Response": {
            "required": True,
            "evidence": "Screenshot/output showing successful endpoint response",
            "status": None
        }
    },
    "STANDOUT_FEATURES": {
        "ONNX Conversion": {
            "required": False,
            "evidence": "Model converted to ONNX format (optional)",
            "status": None
        },
        "IoT Edge Deployment": {
            "required": False,
            "evidence": "Notebook showing edge deployment (optional)",
            "status": None
        },
        "App Insights Logging": {
            "required": False,
            "evidence": "Application Insights metrics and logs collected (optional)",
            "status": None
        }
    }
}

print("✓ Rubric criteria structure created")
print(f"\nTotal criteria: {sum(len(v) for v in rubric_criteria.values())}")
print(f"Categories: {list(rubric_criteria.keys())}")

## 3. Project Artifact Discovery in Repository

In [ ]:
# Scan repository for artifacts
artifact_manifest = {
    "readme": None,
    "src_files": [],
    "config_files": [],
    "environment_files": [],
    "notebooks": [],
    "screenshots": [],
    "artifacts": {}
}

# Find README
readme_path = project_root / "README.md"
if readme_path.exists():
    artifact_manifest["readme"] = str(readme_path)
    print(f"✓ README found: {readme_path}")
else:
    print(f"✗ README not found at {readme_path}")

# Find source files
src_dir = project_root / "src"
if src_dir.exists():
    py_files = list(src_dir.glob("*.py"))
    artifact_manifest["src_files"] = [f.name for f in py_files]
    print(f"\n✓ Found {len(py_files)} source files:")
    for f in sorted(py_files):
        print(f"  - {f.name}")

# Find config files
config_dir = project_root / "config"
if config_dir.exists():
    config_files = list(config_dir.glob("*"))
    artifact_manifest["config_files"] = [f.name for f in config_files]
    print(f"\n✓ Found {len(config_files)} config files:")
    for f in config_files:
        print(f"  - {f.name}")

# Find notebooks
notebooks_dir = project_root / "notebooks"
if notebooks_dir.exists():
    nb_files = list(notebooks_dir.glob("*.ipynb"))
    artifact_manifest["notebooks"] = [f.name for f in nb_files]
    print(f"\n✓ Found {len(nb_files)} notebooks:")
    for f in nb_files:
        print(f"  - {f.name}")

# Find screenshots
if screenshots_dir.exists():
    screenshot_files = list(screenshots_dir.glob("*"))
    artifact_manifest["screenshots"] = [f.name for f in screenshot_files]
    print(f"\n✓ Found {len(screenshot_files)} screenshot files:")
    for f in sorted(screenshot_files)[:5]:  # Show first 5
        print(f"  - {f.name}")
    if len(screenshot_files) > 5:
        print(f"  ... and {len(screenshot_files) - 5} more")

# Find artifacts JSON files
if artifacts_dir.exists():
    json_files = list(artifacts_dir.glob("*.json"))
    for jf in json_files:
        try:
            with open(jf) as f:
                artifact_manifest["artifacts"][jf.name] = json.load(f)
        except:
            artifact_manifest["artifacts"][jf.name] = "(failed to parse)"
    
    print(f"\n✓ Found {len(json_files)} artifact files:")
    for jf in json_files:
        print(f"  - {jf.name}")

print("\n✓ Repository scan complete")

## 4. External Dataset Usage Validation

In [ ]:
# Check for external dataset usage
print("=" * 60)
print("EXTERNAL DATASET VALIDATION")
print("=" * 60)

external_dataset_found = False
expected_dataset_name = "heart-failure-capstone"

# Check if data_prep.py references Kaggle or external source
data_prep_path = project_root / "src" / "data_prep.py"
if data_prep_path.exists():
    with open(data_prep_path) as f:
        content = f.read()
        if "kaggle" in content.lower() or "external" in content.lower() or "url" in content.lower():
            print(f"✓ data_prep.py references external data source")
            external_dataset_found = True

# Check if dataset registered in workspace
if ws:
    try:
        datasets = ws.datasets
        print(f"\n✓ Datasets in workspace: {len(datasets)}")
        for name, dataset in datasets.items():
            print(f"  - {name}: {dataset.id}")
            if expected_dataset_name.lower() in name.lower():
                external_dataset_found = True
                print(f"    ✓ Found expected dataset!")
    except Exception as e:
        print(f"✗ Could not list datasets: {e}")

# Check data_prep.json artifact
data_prep_json = artifacts_dir / "data_prep.json"
if data_prep_json.exists():
    with open(data_prep_json) as f:
        data_prep_info = json.load(f)
        print(f"\n✓ Data preparation artifact found:")
        for key, value in data_prep_info.items():
            print(f"  {key}: {value}")
        external_dataset_found = True

# Set validation status
rubric_criteria["PROJECT_SETUP"]["External Dataset"]["status"] = "PASS" if external_dataset_found else "FAIL"

if external_dataset_found:
    print(f"\n✓✓ EXTERNAL DATASET VALIDATION: PASS")
else:
    print(f"\n✗ EXTERNAL DATASET VALIDATION: FAIL")

## 5. README Requirements Validation

In [ ]:
# Validate README content
print("=" * 60)
print("README REQUIREMENTS VALIDATION")
print("=" * 60)

readme_requirements = {
    "Project Overview": ["overview", "project", "heart failure"],
    "Dataset Description": ["dataset", "kaggle", "records", "features"],
    "Data Ingestion": ["ingestion", "upload", "prepare data", "data prep"],
    "AutoML Settings": ["automl", "configuration", "classification", "primary metric"],
    "HyperDrive Parameters": ["hyperdrive", "hyperparameter", "tuning", "range"],
    "Model Comparison": ["compare", "best model", "accuracy", "winner"],
    "Endpoint Instructions": ["endpoint", "deploy", "request", "inference", "uri"],
    "Future Improvements": ["improvement", "enhancement", "future"],
    "Screenshots": ["screenshot", "png", "jpg"],
    "Screencast": ["screencast", "youtube", "video", "link"]
}

if readme_path.exists():
    with open(readme_path) as f:
        readme_content = f.read().lower()
    
    readme_length = len(readme_content)
    print(f"✓ README size: {readme_length} characters")
    
    print(f"\nREADME Section Checks:")
    for section, keywords in readme_requirements.items():
        found = any(kw in readme_content for kw in keywords)
        status = "✓" if found else "✗"
        print(f"  {status} {section}: {'Found' if found else 'MISSING'}")
        rubric_criteria["PROJECT_SETUP"][f"README: {section}"]["status"] = "PASS" if found else "FAIL"
else:
    print(f"✗ README not found")
    rubric_criteria["PROJECT_SETUP"]["README Exists"]["status"] = "FAIL"

## 6. AutoML Configuration and Run Validation

In [ ]:
# Validate AutoML configuration
print("=" * 60)
print("AUTOML CONFIGURATION VALIDATION")
print("=" * 60)

automl_config_found = False
automl_code_path = project_root / "src" / "automl_run.py"

if automl_code_path.exists():
    with open(automl_code_path) as f:
        content = f.read()
        
    # Check for key AutoML components
    checks = {
        "AutoMLConfig": "AutoMLConfig" in content,
        "Task Type": "classification" in content.lower(),
        "Primary Metric": "primary_metric" in content,
        "Training Data": "training_data" in content,
        "Compute Target": "compute_target" in content,
        "Timeout": "timeout" in content.lower(),
        "Cross Validation": "cross" in content.lower()
    }
    
    print(f"✓ Found automl_run.py\n")
    print("AutoML Configuration Components:")
    for check, found in checks.items():
        status = "✓" if found else "✗"
        print(f"  {status} {check}")
        automl_config_found = automl_config_found or found
else:
    print(f"✗ automl_run.py not found")

# Check for AutoML artifacts
automl_json = artifacts_dir / "automl_results.json"
if automl_json.exists():
    with open(automl_json) as f:
        automl_results = json.load(f)
    print(f"\n✓ AutoML results artifact found:")
    for key, value in automl_results.items():
        print(f"  {key}: {value}")
else:
    print(f"\n  (automl_results.json will be created after execution)")

# Try to retrieve AutoML runs from workspace
if ws:
    try:
        automl_exp = Experiment(ws, name="capstone-automl")
        automl_runs = list(automl_exp.get_runs())
        print(f"\n✓ Found {len(automl_runs)} AutoML runs in workspace")
        if automl_runs:
            print(f"\nLatest AutoML run: {automl_runs[-1].id}")
            print(f"Status: {automl_runs[-1].status}")
    except Exception as e:
        print(f"\n  (No AutoML runs found yet - will be created after execution)")

rubric_criteria["AUTOML_MODEL"]["AutoML Code"]["status"] = "PASS" if automl_code_path.exists() else "FAIL"
rubric_criteria["AUTOML_MODEL"]["AutoML Configuration"]["status"] = "PASS" if automl_config_found else "FAIL"

## 7. AutoML Best Model Validation

In [ ]:
# Validate AutoML best model
print("=" * 60)
print("AUTOML BEST MODEL VALIDATION")
print("=" * 60)

automl_best_model_found = False

# Check artifact
if 'automl_results' in locals():
    print(f"✓ AutoML Results Summary:")
    print(f"  Best Run ID: {automl_results.get('best_child_run_id', 'N/A')}")
    print(f"  Model Name: {automl_results.get('registered_model_name', 'N/A')}")
    print(f"  Model Version: {automl_results.get('registered_model_version', 'N/A')}")
    print(f"  Best Accuracy: {automl_results.get('best_accuracy', 'N/A')}")
    
    # Check if model is registered
    if ws and 'registered_model_name' in automl_results:
        try:
            model = Model(ws, name=automl_results['registered_model_name'])
            print(f"\n✓ Model registered in workspace:")
            print(f"  Name: {model.name}")
            print(f"  ID: {model.id}")
            automl_best_model_found = True
        except:
            pass
else:
    print(f"  (AutoML artifacts will be available after execution)")

rubric_criteria["AUTOML_MODEL"]["Best Model Properties"]["status"] = "PASS" if automl_best_model_found else "UNKNOWN"
rubric_criteria["AUTOML_MODEL"]["Model Registration"]["status"] = "PASS" if automl_best_model_found else "UNKNOWN"

## 8. HyperDrive Configuration Validation

In [ ]:
# Validate HyperDrive configuration
print("=" * 60)
print("HYPERDRIVE CONFIGURATION VALIDATION")
print("=" * 60)

hyperdrive_config_found = False
hyperdrive_code_path = project_root / "src" / "hyperdrive_run.py"
train_script_path = project_root / "src" / "train_hyperdrive.py"

if hyperdrive_code_path.exists():
    with open(hyperdrive_code_path) as f:
        hd_content = f.read()
    
    # Check for key HyperDrive components
    checks = {
        "HyperDriveConfig": "HyperDriveConfig" in hd_content,
        "Sampling Method": any(s in hd_content for s in ["GridParameterSampling", "RandomParameterSampling", "BayesianParameterSampling"]),
        "Parameter Space": "choice" in hd_content or "uniform" in hd_content or "loguniform" in hd_content,
        "Primary Metric": "primary_metric" in hd_content,
        "Max Total Runs": "max_total_runs" in hd_content,
        "Max Concurrent Runs": "max_concurrent_runs" in hd_content
    }
    
    print(f"✓ Found hyperdrive_run.py\n")
    print("HyperDrive Configuration Components:")
    for check, found in checks.items():
        status = "✓" if found else "✗"
        print(f"  {status} {check}")
        hyperdrive_config_found = hyperdrive_config_found or found
    
    # Detect sampling method
    if "GridParameterSampling" in hd_content:
        print(f"\n  Sampling Method: Grid Sampling")
    elif "RandomParameterSampling" in hd_content:
        print(f"\n  Sampling Method: Random Sampling")
    elif "BayesianParameterSampling" in hd_content:
        print(f"\n  Sampling Method: Bayesian Sampling")
else:
    print(f"✗ hyperdrive_run.py not found")

if train_script_path.exists():
    with open(train_script_path) as f:
        train_content = f.read()
    
    print(f"\n✓ Found train_hyperdrive.py")
    if "run.log" in train_content:
        print(f"  ✓ Metrics logging detected")
else:
    print(f"\n✗ train_hyperdrive.py not found")

rubric_criteria["HYPERDRIVE_MODEL"]["HyperDrive Code"]["status"] = "PASS" if hyperdrive_code_path.exists() else "FAIL"
rubric_criteria["HYPERDRIVE_MODEL"]["Sampling Method"]["status"] = "PASS" if hyperdrive_config_found else "FAIL"

## 9. HyperDrive Best Model Validation

In [ ]:
# Validate HyperDrive best model
print("=" * 60)
print("HYPERDRIVE BEST MODEL VALIDATION")
print("=" * 60)

hyperdrive_best_model_found = False

# Check artifact
hyperdrive_json = artifacts_dir / "hyperdrive_results.json"
if hyperdrive_json.exists():
    with open(hyperdrive_json) as f:
        hyperdrive_results = json.load(f)
    print(f"✓ HyperDrive Results Summary:")
    print(f"  Best Run ID: {hyperdrive_results.get('best_run_id', 'N/A')}")
    print(f"  Model Name: {hyperdrive_results.get('registered_model_name', 'N/A')}")
    print(f"  Best Accuracy: {hyperdrive_results.get('best_accuracy', 'N/A')}")
    print(f"  Best Hyperparameters: {hyperdrive_results.get('best_hyperparameters', 'N/A')}")
    
    # Check if model is registered
    if ws and 'registered_model_name' in hyperdrive_results:
        try:
            model = Model(ws, name=hyperdrive_results['registered_model_name'])
            print(f"\n✓ Model registered in workspace:")
            print(f"  Name: {model.name}")
            print(f"  ID: {model.id}")
            hyperdrive_best_model_found = True
        except:
            pass
else:
    print(f"  (HyperDrive artifacts will be available after execution)")

rubric_criteria["HYPERDRIVE_MODEL"]["Best Hyperparameters"]["status"] = "PASS" if hyperdrive_best_model_found else "UNKNOWN"
rubric_criteria["HYPERDRIVE_MODEL"]["Model Registration"]["status"] = "PASS" if hyperdrive_best_model_found else "UNKNOWN"

## 10. Model Comparison and Deployment Validation

In [ ]:
# Validate model comparison
print("=" * 60)
print("MODEL COMPARISON VALIDATION")
print("=" * 60)

comparison_json = artifacts_dir / "comparison_results.json"
if comparison_json.exists():
    with open(comparison_json) as f:
        comparison_results = json.load(f)
    print(f"✓ Model Comparison Results:")
    print(f"  AutoML Accuracy: {comparison_results.get('automl_accuracy', 'N/A')}")
    print(f"  HyperDrive Accuracy: {comparison_results.get('hyperdrive_accuracy', 'N/A')}")
    print(f"  Winner: {comparison_results.get('winner', 'N/A')}")
    print(f"  Winning Accuracy: {comparison_results.get('winning_accuracy', 'N/A')}")
else:
    print(f"  (Model comparison will be available after execution)")

## 11. Deployment and Endpoint Validation

In [ ]:
# Validate deployment
print("=" * 60)
print("DEPLOYMENT VALIDATION")
print("=" * 60)

endpoint_active = False
deployment_json = artifacts_dir / "deployment_details.json"

if deployment_json.exists():
    with open(deployment_json) as f:
        deployment_info = json.load(f)
    print(f"✓ Deployment Details:")
    print(f"  Service Name: {deployment_info.get('service_name', 'N/A')}")
    print(f"  State: {deployment_info.get('state', 'N/A')}")
    print(f"  Scoring URI: {deployment_info.get('scoring_uri', 'N/A')}")
    print(f"  Swagger URI: {deployment_info.get('swagger_uri', 'N/A')}")
    print(f"  App Insights: {deployment_info.get('app_insights_enabled', 'N/A')}")
    
    endpoint_active = deployment_info.get('state', '').lower() == 'active'
else:
    print(f"  (Deployment details will be available after execution)")

# Check endpoint in workspace
if ws:
    try:
        service = Webservice(ws, name="capstone-endpoint")
        print(f"\n✓ Endpoint found in workspace:")
        print(f"  Name: {service.name}")
        print(f"  State: {service.state}")
        print(f"  Scoring URI: {service.scoring_uri}")
        endpoint_active = service.state == "Healthy"
    except:
        pass

rubric_criteria["DEPLOYMENT"]["Model Deployment"]["status"] = "PASS" if deployment_json.exists() else "UNKNOWN"
rubric_criteria["DEPLOYMENT"]["Endpoint Active"]["status"] = "PASS" if endpoint_active else "UNKNOWN"

## 12. Inference Request Validation

In [ ]:
# Validate inference request
print("=" * 60)
print("INFERENCE REQUEST VALIDATION")
print("=" * 60)

inference_test_passed = False
consume_json = artifacts_dir / "consume_results.json"

if consume_json.exists():
    with open(consume_json) as f:
        consume_results = json.load(f)
    print(f"✓ Inference Test Results:")
    print(f"  Endpoint: {consume_results.get('endpoint', 'N/A')}")
    print(f"  Status Code: {consume_results.get('status_code', 'N/A')}")
    print(f"  Response: {consume_results.get('response', 'N/A')}")
    
    inference_test_passed = consume_results.get('status_code') == 200
else:
    print(f"  (Inference test will be available after execution)")

rubric_criteria["DEPLOYMENT"]["Inference Request Code"]["status"] = "PASS" if (project_root / "src" / "consume.py").exists() else "FAIL"
rubric_criteria["DEPLOYMENT"]["Inference Response"]["status"] = "PASS" if inference_test_passed else "UNKNOWN"

## 13. Standout Features Validation

In [ ]:
# Check for standout features
print("=" * 60)
print("STANDOUT FEATURES VALIDATION")
print("=" * 60)

onnx_found = False
iot_edge_found = False
app_insights_found = False

# Check for ONNX conversion
if notebooks_dir.exists():
    for nb in notebooks_dir.glob("*.ipynb"):
        with open(nb) as f:
            try:
                nb_content = f.read()
                if "onnx" in nb_content.lower() or "convert" in nb_content.lower():
                    onnx_found = True
            except:
                pass

# Check App Insights in deployment code
if (project_root / "src" / "deploy.py").exists():
    with open(project_root / "src" / "deploy.py") as f:
        deploy_content = f.read()
        if "app_insights" in deploy_content.lower() or "application_insights" in deploy_content.lower():
            app_insights_found = True

print(f"Optional Standout Features:")
print(f"  {'✓' if onnx_found else '✗'} ONNX Conversion")
print(f"  {'✓' if iot_edge_found else '✗'} IoT Edge Deployment")
print(f"  {'✓' if app_insights_found else '✗'} App Insights Logging")

rubric_criteria["STANDOUT_FEATURES"]["ONNX Conversion"]["status"] = "PASS" if onnx_found else "NOT IMPLEMENTED"
rubric_criteria["STANDOUT_FEATURES"]["IoT Edge Deployment"]["status"] = "PASS" if iot_edge_found else "NOT IMPLEMENTED"
rubric_criteria["STANDOUT_FEATURES"]["App Insights Logging"]["status"] = "PASS" if app_insights_found else "NOT IMPLEMENTED"

## 14. Compliance Scorecard

In [ ]:
# Generate compliance scorecard
print("=" * 80)
print("CAPSTONE PROJECT COMPLIANCE SCORECARD")
print("=" * 80)

# Build compliance report
compliance_report = []
total_criteria = 0
pass_count = 0
fail_count = 0
unknown_count = 0

for category, criteria in rubric_criteria.items():
    print(f"\n{category}")
    print("-" * 80)
    
    for criterion, details in criteria.items():
        total_criteria += 1
        status = details["status"] or "NOT CHECKED"
        required = "*REQUIRED*" if details["required"] else "(optional)"
        
        if status == "PASS":
            symbol = "✓"
            pass_count += 1
        elif status == "FAIL":
            symbol = "✗"
            fail_count += 1
        else:
            symbol = "?"
            unknown_count += 1
        
        print(f"  {symbol} {criterion:<40} {status:<20} {required}")
        
        compliance_report.append({
            "Category": category,
            "Criterion": criterion,
            "Status": status,
            "Required": details["required"],
            "Evidence": details["evidence"]
        })

print("\n" + "=" * 80)
print(f"SUMMARY")
print("=" * 80)
print(f"Total Criteria: {total_criteria}")
print(f"✓ Passed: {pass_count}")
print(f"✗ Failed: {fail_count}")
print(f"? Unknown/Not Checked: {unknown_count}")
print(f"\nCompletion: {pass_count}/{total_criteria} ({100*pass_count//total_criteria}%)")

if fail_count == 0 and unknown_count == 0:
    print(f"\n🎉 PROJECT READY FOR SUBMISSION 🎉")
elif fail_count == 0:
    print(f"\n⚠️  Awaiting execution to validate {unknown_count} criteria")
else:
    print(f"\n⚠️  {fail_count} criteria require attention")

## 15. Missing Items Report Export

In [ ]:
# Generate missing items report
print("=" * 80)
print("MISSING ITEMS REPORT")
print("=" * 80)

missing_items = [
    item for item in compliance_report 
    if item["Required"] and item["Status"] != "PASS"
]

if missing_items:
    print(f"\nFound {len(missing_items)} missing REQUIRED items:\n")
    for i, item in enumerate(missing_items, 1):
        print(f"{i}. [{item['Category']}] {item['Criterion']}")
        print(f"   Status: {item['Status']}")
        print(f"   Evidence Needed: {item['Evidence']}")
        print()
else:
    print(f"\n✓ No missing REQUIRED items!")

# Save report to file
report_path = artifacts_dir / "rubric_compliance_report.json"
with open(report_path, "w") as f:
    json.dump({
        "timestamp": datetime.now().isoformat(),
        "total_criteria": total_criteria,
        "pass_count": pass_count,
        "fail_count": fail_count,
        "unknown_count": unknown_count,
        "missing_required": missing_items,
        "all_criteria": compliance_report
    }, f, indent=2)

print(f"\n✓ Compliance report saved to: {report_path}")

## Notebook Complete

This validation notebook checks the Capstone project against all Udacity rubric criteria.

**Next Steps**:
1. Execute all cells in this notebook
2. Run the 6 main Python scripts (data_prep → automl → hyperdrive → compare → deploy → consume)
3. Re-run this notebook to validate execution results
4. Check the `rubric_compliance_report.json` artifact
5. Address any missing items
6. Complete screencast recording and README updates